# Lesson 6: Embeddings

For each cell: **guess what it will print first**, then press ▶ and check.

From [Build a Tiny LLM](https://gradlabs.in/learn/tiny-llm/lessons/lesson6.html).

**Setup.** Loads PyTorch and our text, and gives each character a number.

In [ ]:
import torch
import torch.nn.functional as F

text = "the cat sat on the mat. the dog sat on the log."
unique_chars = sorted(set(text))
vocab_size = len(unique_chars)
char_to_number = {}
number_to_char = {}
for number, char in enumerate(unique_chars):
    char_to_number[char] = number
    number_to_char[number] = char

**Examples.** Every example is the last 3 letters and the letter that came next. There are 44 of them.

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# Examples: the last 3 letters, and the next one
third_last = []
second_last = []
last = []
next_chars = []
for i in range(len(text) - 3):
    third_last.append(char_to_number[text[i]])
    second_last.append(char_to_number[text[i + 1]])
    last.append(char_to_number[text[i + 2]])
    next_chars.append(char_to_number[text[i + 3]])
third_last = torch.tensor(third_last)
second_last = torch.tensor(second_last)
last = torch.tensor(last)
next_chars = torch.tensor(next_chars)
print("Examples:", len(next_chars))

**The model: embeddings and weights.** The embedding table gives every letter 2 numbers. The weights say how much each of the 6 numbers counts for each next letter. 112 numbers in all.

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# The model: 2 embedding numbers per letter, 6 weights per next letter
torch.manual_seed(42)
embedding = torch.randn((vocab_size, 2), requires_grad=True)
weights = torch.randn((6, vocab_size), requires_grad=True)
print("Numbers in the model:", embedding.numel() + weights.numel())

**The model learns.** Put the 3 letters' embeddings side by side (6 numbers), multiply by the weights to get the scores, then guess, measure, find directions, move. The push reaches the embeddings too.

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# Learn: predict, measure, find directions, move
learning_rate = 1
for step in range(300):
    numbers = torch.cat([embedding[third_last], embedding[second_last], embedding[last]], dim=1)   # a) the 6 embedding numbers of every example
    scores = numbers @ weights                                                                     # multiply by the weights, add up
    loss = F.cross_entropy(scores, next_chars)                                                     # b) measure
    embedding.grad = None
    weights.grad = None
    loss.backward()                                                                                # c) find directions
    embedding.data = embedding.data - learning_rate * embedding.grad                               # d) move
    weights.data = weights.data - learning_rate * weights.grad
    if step % 50 == 0:
        print("Step", step, "loss:", round(loss.item(), 3))

**Write new text.** Start from “the” and pick 40 letters, each from the chances after the last 3 letters.

🤔 *Guess what this prints, then press ▶.*

In [ ]:
# Write 40 new letters, starting from "the"
torch.manual_seed(42)
output = "the"
for step in range(40):
    numbers = torch.cat([embedding[char_to_number[output[-3]]], embedding[char_to_number[output[-2]]], embedding[char_to_number[output[-1]]]])
    scores = numbers @ weights
    chances = F.softmax(scores, dim=0)
    picked = torch.multinomial(chances, num_samples=1).item()
    output = output + number_to_char[picked]
print(output)